In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [2]:
input_str = 'apple'
label_str = 'pple!'
char_vocab = sorted(list(set(input_str + label_str)))
# set(input_str + label_str) 는 input_str과 label_str을 합친 후 중복을 제거함. 그러면 a p l e ! 가 남음
# 그리고 sorted(list) 는 리스트로 바꾼 후, sorted를 사용해 ASCII순서를 가지게 배열됨
vocab_size = len(char_vocab) # 고유 문자수를 계산 len(char_vocab)
print('문자 집합의 크기 : {}'.format(vocab_size))
# 'a', 'p', 'l', 'e', '!' 
print(char_vocab)

문자 집합의 크기 : 5
['!', 'a', 'e', 'l', 'p']


In [3]:
input_size = vocab_size
hidden_size = 5
output_size = 5
learning_rate = 0.1

In [4]:
char_to_index = dict((c, i) for i , c in enumerate(char_vocab)) 
print(char_to_index)

{'!': 0, 'a': 1, 'e': 2, 'l': 3, 'p': 4}


In [5]:
index_to_char = {}
for key, value in char_to_index.items():
    index_to_char[value] = key
print(index_to_char)

{0: '!', 1: 'a', 2: 'e', 3: 'l', 4: 'p'}


In [6]:
x_data = [char_to_index[c] for c in input_str]
y_data = [char_to_index[c] for c in label_str]
print(x_data) # a, p, p, l, e
print(y_data) # p, p, l, e, !

[1, 4, 4, 3, 2]
[4, 4, 3, 2, 0]


In [7]:
#nn.RNN() 는 3차원 tensor를 입력받으므로 batch dimension 추가
x_data = [x_data]
y_data = [y_data]
print(x_data)
print(y_data)

[[1, 4, 4, 3, 2]]
[[4, 4, 3, 2, 0]]


In [8]:
x_one_hot = [np.eye(vocab_size)[x] for x in x_data]
print(x_one_hot)

[array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.]])]


In [9]:
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)
print(X)
print(Y)
print('X의 크기 : {}'.format(X.shape))
print('Y의 크기 : {}'.format(Y.shape))

tensor([[[0., 1., 0., 0., 0.],
         [0., 0., 0., 0., 1.],
         [0., 0., 0., 0., 1.],
         [0., 0., 0., 1., 0.],
         [0., 0., 1., 0., 0.]]])
tensor([[4, 4, 3, 2, 0]])
X의 크기 : torch.Size([1, 5, 5])
Y의 크기 : torch.Size([1, 5])


/tmp/ipython-input-4212589629.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  X = torch.FloatTensor(x_one_hot)


In [10]:
class Net(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_size, hidden_size, batch_first= True)
        self.fc = torch.nn.Linear(hidden_size, output_size, bias=True)
        self.fc = torch.nn.Linear(hidden_size, output_size, bias=True)

    def forward(self, x):
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x


In [11]:
net = Net(input_size, hidden_size, output_size)

In [12]:
outputs = net(X)
print(outputs.shape)

torch.Size([1, 5, 5])


In [13]:
print(outputs.view(-1, input_size).shape)

torch.Size([5, 5])


In [14]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([1, 5])
torch.Size([5])


In [15]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [16]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X)
    loss = criterion(outputs.view(-1, input_size), Y.view(-1))
    loss.backward()
    optimizer.step()

    result = outputs.data.numpy().argmax(axis=2)

    result_str = ''.join([index_to_char[c] for c in np.squeeze(result)])
    print(i, "loss : ", loss.item(), "prediction : ", result, "true Y : ", y_data, "prediction str : ", result_str)


0 loss :  1.6078062057495117 prediction :  [[3 3 3 3 3]] true Y :  [[4, 4, 3, 2, 0]] prediction str :  lllll
1 loss :  1.342104434967041 prediction :  [[4 4 4 4 3]] true Y :  [[4, 4, 3, 2, 0]] prediction str :  ppppl
2 loss :  1.1318107843399048 prediction :  [[4 4 4 4 0]] true Y :  [[4, 4, 3, 2, 0]] prediction str :  pppp!
3 loss :  0.9002559781074524 prediction :  [[4 4 4 2 0]] true Y :  [[4, 4, 3, 2, 0]] prediction str :  pppe!
4 loss :  0.6708110570907593 prediction :  [[4 4 4 2 0]] true Y :  [[4, 4, 3, 2, 0]] prediction str :  pppe!
5 loss :  0.4804997444152832 prediction :  [[4 4 4 2 0]] true Y :  [[4, 4, 3, 2, 0]] prediction str :  pppe!
6 loss :  0.3335891366004944 prediction :  [[4 4 3 2 0]] true Y :  [[4, 4, 3, 2, 0]] prediction str :  pple!
7 loss :  0.2247360646724701 prediction :  [[4 4 3 2 0]] true Y :  [[4, 4, 3, 2, 0]] prediction str :  pple!
8 loss :  0.15099957585334778 prediction :  [[4 4 3 2 0]] true Y :  [[4, 4, 3, 2, 0]] prediction str :  pple!
9 loss :  0.1012909